# Pixel Maps for Rossby Number, Speed, and Tchla

This notebook plots a single-day spatial snapshot from bronze and silver data. Rossby number and speed come from the bronze SWOT L4 grid. Tchla comes from silver pigment pixels inside tracked eddy contours. The bottom histogram shows the Rossby number interpolated from the SWOT grid to those silver pigment pixels, split by eddy polarity.

In [ ]:
from pathlib import Path
import datetime as dt
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.colors import TwoSlopeNorm

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
EXPERIMENT = "gulf_stream_pigment_influencers_20241001_20251231"
PLOT_DATE = pd.Timestamp("2025-08-16")

swot_dir = repo_root / "data" / EXPERIMENT / "bronze" / "swot_l4"
pigment_dir = repo_root / "data" / EXPERIMENT / "silver" / "pigments"
swot_date_re = re.compile(r"(\d{8})")

def parse_swot_date(path):
    match = swot_date_re.search(path.name)
    if match is None:
        return None
    return pd.Timestamp(dt.datetime.strptime(match.group(1), "%Y%m%d").date())

swot_files = [(parse_swot_date(path), path) for path in sorted(swot_dir.glob("*.nc"))]
swot_files = [(date, path) for date, path in swot_files if date is not None]
swot_date, swot_path = min(swot_files, key=lambda item: abs((item[0] - PLOT_DATE).days))

swot_date, swot_path.name


In [ ]:
pigment_frames = []
for polarity in ("cyclone", "anticyclone"):
    for path in sorted((pigment_dir / polarity).glob("eddy_*_pigments.parquet")):
        df = pd.read_parquet(path, columns=["track_id", "date", "pixel_lon", "pixel_lat", "T chla"])
        df["date"] = pd.to_datetime(df["date"])
        df = df[df["date"] == PLOT_DATE]
        if df.empty:
            continue
        df["polarity"] = polarity
        pigment_frames.append(df)

if not pigment_frames:
    raise ValueError(f"No silver pigment pixels found for {PLOT_DATE.date()}")

pigments = pd.concat(pigment_frames, ignore_index=True)
lon_pad = 1.0
lat_pad = 1.0
xlim = (pigments["pixel_lon"].min() - lon_pad, pigments["pixel_lon"].max() + lon_pad)
ylim = (pigments["pixel_lat"].min() - lat_pad, pigments["pixel_lat"].max() + lat_pad)

print(f"PACE pigment date: {PLOT_DATE.date()} ({len(pigments):,} silver pixels)")
print(f"SWOT file date: {swot_date.date()} ({swot_path.name})")
pigments.groupby("polarity")["T chla"].agg(["count", "min", "max", "mean", "median"])


In [ ]:
with xr.open_dataset(swot_path) as ds:
    if "time" in ds["relative_vorticity"].dims:
        ds = ds.isel(time=0)
    region = ds.sel(longitude=slice(xlim[0], xlim[1]), latitude=slice(ylim[0], ylim[1]))
    lon = region["longitude"].to_numpy()
    lat = region["latitude"].to_numpy()
    rossby_number = region["relative_vorticity"].to_numpy()
    speed = np.hypot(region["ugos"].to_numpy(), region["vgos"].to_numpy())

    full_rossby = ds["relative_vorticity"]
    pigments["rossby_at_pixel"] = full_rossby.interp(
        longitude=xr.DataArray(pigments["pixel_lon"].to_numpy(), dims="point"),
        latitude=xr.DataArray(pigments["pixel_lat"].to_numpy(), dims="point"),
    ).to_numpy()

rossby_range = pigments.groupby("polarity")["rossby_at_pixel"].agg(["count", "min", "max", "mean", "median"])
rossby_range


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 20))
map_axes = axes[:3]
hist_ax = axes[3]

rossby_abs = np.nanmax(np.abs(rossby_number))
rossby_norm = TwoSlopeNorm(vmin=-rossby_abs, vcenter=0, vmax=rossby_abs)
rossby_mesh = map_axes[0].pcolormesh(lon, lat, rossby_number, shading="auto", cmap="coolwarm", norm=rossby_norm)
map_axes[0].set_title(f"Bronze SWOT Rossby number, Ro = zeta/f ({swot_date.date()})")
fig.colorbar(rossby_mesh, ax=map_axes[0], label="Rossby number, Ro")

speed_mesh = map_axes[1].pcolormesh(lon, lat, speed, shading="auto", cmap="viridis")
map_axes[1].set_title("Bronze SWOT geostrophic speed from ugos/vgos")
fig.colorbar(speed_mesh, ax=map_axes[1], label="Speed (m s$^{-1}$)")

tchla_scatter = map_axes[2].scatter(
    pigments["pixel_lon"],
    pigments["pixel_lat"],
    c=pigments["T chla"],
    s=5,
    marker="s",
    linewidths=0,
    cmap="YlGn",
    alpha=0.85,
)
map_axes[2].set_title("Silver pigment pixels: Tchla inside tracked eddies")
fig.colorbar(tchla_scatter, ax=map_axes[2], label="Tchla (mg m$^{-3}$)")

for ax in map_axes:
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.25, linestyle="--")

map_axes[2].set_xlabel("Longitude")

hist_values = pigments["rossby_at_pixel"].dropna()
bins = np.linspace(hist_values.min(), hist_values.max(), 35)
polarity_colors = {"cyclone": "#4c72b0", "anticyclone": "#c44e52"}
for polarity in ("cyclone", "anticyclone"):
    vals = pigments.loc[pigments["polarity"] == polarity, "rossby_at_pixel"].dropna()
    hist_ax.hist(
        vals,
        bins=bins,
        alpha=0.65,
        color=polarity_colors[polarity],
        edgecolor="white",
        label=f"{polarity}: {vals.min():.3f} to {vals.max():.3f}",
    )

hist_ax.axvline(0, color="black", lw=1, alpha=0.7)
hist_ax.set_title("Rossby number at silver pigment pixels by eddy polarity")
hist_ax.set_xlabel("Interpolated Rossby number, Ro")
hist_ax.set_ylabel("Pixel count")
hist_ax.grid(axis="y", alpha=0.25, linestyle="--")
hist_ax.legend(loc="upper right")

fig.suptitle(f"Pixel/grid maps for {PLOT_DATE.date()}", fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()
